In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
Telugu Manuscript Line Reconstruction - CNN Encoder + Transformer Decoder
===========================================================================
Reads a (possibly damaged) Telugu manuscript line image and outputs the
reconstructed Telugu text. Works across all 8 of your damage types (clean,
gaps, stains, fade, degraded, mixed) since the SAME model is trained on
all of them pooled together - the encoder learns to read through visual
noise, the decoder learns to guess fully-missing spans from context.

HOW TO USE IN KAGGLE:
1. Edit the CONFIG section below to point at your CSV + image folder.
2. Your CSV must have two columns: one with the image filename/path,
   one with the ground-truth Telugu text for that line.
3. Run this script top to bottom. With smoke_test=True it trains on a
   small slice of your data for a few epochs, just to prove the whole
   pipeline runs without crashing and the loss goes down.
4. Once that passes, set smoke_test=False and raise epochs for a real run.
"""

import os
import re
import math
import random
import time  # 1. Track Total & Epoch Training Time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# --- 1 RANDOM SEED ---
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------- CONFIG --------------------------------------
CONFIG = {
    # Folder containing your images AND their matching .txt ground-truth
    # files (e.g. chaos_mixed_00003.png + chaos_mixed_00003.txt side by
    # side). If train/test are separate folders, point this at TRAIN.
    "image_dir": "/kaggle/input/datasets/yvenkatasriramreddy/final-telugu-manuscript-data/10k_Data/train",   
    "test_image_dir": "/kaggle/input/datasets/yvenkatasriramreddy/final-telugu-manuscript-data/10k_Data/validation",  
    "image_extensions": (".png", ".jpg", ".jpeg"),
    "text_extension": ".txt",
    "img_height": 48,          # every line image is resized to this height
    "img_max_width": "auto",   # "auto" = measured from YOUR real images, see below
                                # (set a number only if you want to override it)
    "batch_size": 8,           # your lines are very wide (~5300px) -> keep this modest
    "epochs": 60,              # bump this freely while iterating on the smoke subset
    "smoke_test": False,        # True = quick run on a small subset to verify it runs
    "smoke_test_samples": 300,
    "lr": 3e-4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


# ----------------------------- DATA SCAN ------------------------------------
def scan_dataset(image_dir, image_extensions, text_extension):
    """Walks image_dir (including subfolders, in case you split by type into
    separate folders) and pairs every image with a same-named .txt file.
    Also extracts a 'type' tag from the filename (e.g. chaos_mixed_00003.png
    -> type 'chaos_mixed') purely so you can print a per-type count and
    confirm your 8 types really are balanced before training."""
    samples = []
    for root, _, files in os.walk(image_dir):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in image_extensions:
                continue
            stem = os.path.splitext(fname)[0]
            txt_path = os.path.join(root, stem + text_extension)
            if not os.path.exists(txt_path):
                continue  # no matching ground-truth file, skip
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read().strip()
            if not text:
                continue
            type_tag = re.sub(r"_\d+$", "", stem)  # strip trailing _00003
            samples.append({
                "image_path": os.path.join(root, fname),
                "text": text,
                "type": type_tag,
            })
    return samples


def compute_max_width(samples, img_h):
    """Scans your ACTUAL images (just reading dimensions, not full pixel
    data, so this is fast) to find how wide the widest line becomes once
    every image is resized to img_h. This is what stops us from guessing
    a width like 512 and silently chopping off most of a 5306px-wide line."""
    max_w = 0
    for s in samples:
        with Image.open(s["image_path"]) as img:
            w, h = img.size
        new_w = max(1, int(w * (img_h / h)))
        max_w = max(max_w, new_w)
    # round up to a multiple of 8 so the CNN's three 2x poolings (8x total
    # downsampling) divide the width evenly with no rounding weirdness
    max_w = ((max_w + 7) // 8) * 8
    return max_w


# ----------------------------- VOCAB ----------------------------------------
class Vocab:
    """Character-level vocabulary built straight from your training texts.
    NOTE: this treats every Unicode codepoint (base consonant, vowel sign,
    virama, etc.) as its own token. That's simpler to get running today.
    A natural upgrade later is grapheme-cluster tokens, matching the
    character segmentation you already did, but codepoint-level works
    fine to get a model training and is what this script uses."""

    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
        self.itos = specials + chars
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.sos_id = self.stoi["<sos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        ids = [self.sos_id] + [self.stoi.get(ch, self.unk_id) for ch in text] + [self.eos_id]
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos_id:
                break
            if i in (self.pad_id, self.sos_id):
                continue
            out.append(self.itos[i])
        return "".join(out)


# ----------------------------- DATASET ---------------------------------------
class TeluguLineDataset(Dataset):
    """One sample = one manuscript line image (any of your 8 damage types)
    + its ground-truth Telugu text, read from a same-named .txt file."""

    def __init__(self, samples, vocab, img_h, img_w):
        self.samples = samples
        self.vocab = vocab
        self.img_h = img_h
        self.img_w = img_w
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        img = Image.open(path).convert("L")          # grayscale
        w, h = img.size
        new_w = max(1, int(w * (self.img_h / h)))
        img = img.resize((new_w, self.img_h))
        img = self.to_tensor(img)                      # [1, H, W'] in [0,1]
        c, h_, w_ = img.shape
        if w_ < self.img_w:                             # pad to fixed width
            pad = torch.ones(c, h_, self.img_w - w_)
            img = torch.cat([img, pad], dim=2)
        else:                                            # crop if too wide
            img = img[:, :, : self.img_w]
        return img

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = self._load_image(sample["image_path"])
        target = self.vocab.encode(sample["text"])
        return image, target


def collate_fn(batch, pad_id):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    max_len = max(t.size(0) for t in targets)
    padded = torch.full((len(targets), max_len), pad_id, dtype=torch.long)
    for i, t in enumerate(targets):
        padded[i, : t.size(0)] = t
    return images, padded


# ----------------------------- MODEL ------------------------------------------
class CNNEncoder(nn.Module):
    """Turns an image [B, 1, H, W] into a left-to-right sequence of feature
    vectors - one per horizontal slice, the same way a human eye scans a
    line of text. This is the part that has to read through stains/fade."""

    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(),
        )
        self.height_pool = nn.AdaptiveAvgPool2d((1, None))  # collapse height to 1

    def forward(self, x):
        feat = self.conv(x)            # [B, d_model, H', W']
        feat = self.height_pool(feat)  # [B, d_model, 1, W']
        feat = feat.squeeze(2)         # [B, d_model, W']
        feat = feat.permute(0, 2, 1)   # [B, W', d_model]  (sequence-first)
        return feat


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TeluguOCRModel(nn.Module):
    """CNN encoder + Transformer decoder. The decoder is what lets the
    model GUESS characters that are completely missing (your Type 2 gaps,
    and the gap-y parts of Type 7/8) by leaning on learned Telugu language
    patterns instead of needing visual evidence for every character."""

    def __init__(self, vocab_size, d_model=256, nhead=8, num_decoder_layers=4,
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.pos_enc_src = PositionalEncoding(d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc_tgt = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, images, tgt_in):
        memory = self.pos_enc_src(self.encoder(images))      # [B, S, d_model]
        tgt_emb = self.pos_enc_tgt(self.tgt_embed(tgt_in))    # [B, T, d_model]
        T_len = tgt_in.size(1)
        causal_mask = torch.triu(
            torch.ones(T_len, T_len, device=tgt_in.device), diagonal=1
        ).bool()
        out = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)  # [B, T, d_model]
        return self.fc_out(out)                                    # [B, T, vocab]

    @torch.no_grad()
    def generate(self, images, vocab, max_len=512):
        self.eval()
        memory = self.pos_enc_src(self.encoder(images))
        B = images.size(0)
        ys = torch.full((B, 1), vocab.sos_id, dtype=torch.long, device=images.device)
        for _ in range(max_len):
            tgt_emb = self.pos_enc_tgt(self.tgt_embed(ys))
            T_len = ys.size(1)
            causal_mask = torch.triu(
                torch.ones(T_len, T_len, device=images.device), diagonal=1
            ).bool()
            out = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)
            next_id = self.fc_out(out[:, -1, :]).argmax(-1, keepdim=True)
            ys = torch.cat([ys, next_id], dim=1)
            if (next_id == vocab.eos_id).all():
                break
        return ys


def levenshtein(a, b):
    """Edit distance between two strings - used below to turn predictions
    into an actual accuracy percentage instead of eyeballing one example."""
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = prev if a[i - 1] == b[j - 1] else 1 + min(prev, dp[j - 1], dp[j])
            prev = cur
    return dp[m]


@torch.no_grad()
def evaluate(model, dataset, vocab, device, max_samples=200):
    """Returns (character accuracy %, exact line-match %) on up to
    max_samples from the given dataset. Character accuracy = 1 minus the
    edit-distance rate, which is the standard OCR metric (CRR) and is
    what your '70% accuracy' target should be measured against."""
    model.eval()
    n = min(max_samples, len(dataset))
    indices = random.sample(range(len(dataset)), n)
    total_chars, total_edits, exact = 0, 0, 0
    for idx in indices:
        img, tgt = dataset[idx]
        gt_text = vocab.decode(tgt.tolist())
        pred_ids = model.generate(img.unsqueeze(0).to(device), vocab)[0].tolist()
        pred_text = vocab.decode(pred_ids)
        total_edits += levenshtein(pred_text, gt_text)
        total_chars += max(len(gt_text), 1)
        exact += int(pred_text == gt_text)
    char_acc = 100 * (1 - total_edits / total_chars)
    exact_acc = 100 * exact / n
    return char_acc, exact_acc


# ----------------------------- TRAIN -------------------------------------------
def train_one_run(config):
    samples = scan_dataset(config["image_dir"], config["image_extensions"], config["text_extension"])
    if not samples:
        raise RuntimeError(
            f"No image+.txt pairs found under {config['image_dir']}. "
            f"Check the path and that each image has a same-named .txt next to it."
        )

    # show per-type counts so you can eyeball whether your 8 types are balanced
    type_counts = {}
    for s in samples:
        type_counts[s["type"]] = type_counts.get(s["type"], 0) + 1
    print(f"Found {len(samples)} image+text pairs across {len(type_counts)} type(s):")
    for t, c in sorted(type_counts.items()):
        print(f"  {t}: {c}")

    if config["smoke_test"]:
        n = min(config["smoke_test_samples"], len(samples))
        samples = random.sample(samples, n)
        print(f"\n[SMOKE TEST] Using {len(samples)} samples just to verify the pipeline runs.")

    if config["img_max_width"] in (None, "auto"):
        img_max_width = compute_max_width(samples, config["img_height"])
        print(f"Auto-detected width: your widest line becomes {img_max_width}px "
              f"after resizing height to {config['img_height']}px. Using that.")
    else:
        img_max_width = config["img_max_width"]

    vocab = Vocab([s["text"] for s in samples])
    print(f"Vocab size: {len(vocab)}")

    dataset = TeluguLineDataset(samples, vocab, config["img_height"], img_max_width)
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=True,
        collate_fn=lambda b: collate_fn(b, vocab.pad_id),
    )

    # --- INITIALIZE TEST DATASET ONCE TO PREVENT DISK-SCAN SLOWDOWNS ---
    test_dataset = None
    if config.get("test_image_dir"):
        test_samples = scan_dataset(config["test_image_dir"], config["image_extensions"], config["text_extension"])
        if test_samples:
            if config["smoke_test"]:
                test_samples = test_samples[: min(100, len(test_samples))]
            test_dataset = TeluguLineDataset(test_samples, vocab, config["img_height"], img_max_width)

    device = config["device"]
    model = TeluguOCRModel(vocab_size=len(vocab)).to(device)
    
    # 3. Parameter Count
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n====================")
    print("MODEL INFO")
    print("====================")
    print(f"Total Params     : {total_params:,}")
    print(f"Trainable Params : {trainable_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)

    # --- TRACK BEST MODEL VIA TEST CRR ---
    best_crr = 0
    best_epoch = 0  # 5. Best Epoch Tracking

    # 1. Track Total Training Time (Start)
    training_start = time.time()

    model.train()
    for epoch in range(config["epochs"]):
        epoch_start = time.time()  # 2. Track Epoch Time (Start)
        total_loss = 0.0
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            tgt_in = targets[:, :-1]   # decoder input (shifted right)
            tgt_out = targets[:, 1:]   # what it should predict

            logits = model(images, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{config['epochs']}  loss={total_loss / len(loader):.4f}")
        
        # 2. Track Epoch Time (End & Output)
        epoch_end = time.time()
        print(f"Epoch Time: {(epoch_end - epoch_start):.2f} sec")

        # Evaluate and checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            print("\n--- Running Evaluation ---")
            
            # Evaluate on Test dataset if available to guide early-stopping/saving decisions
            if test_dataset is not None:
                char_acc, exact_acc = evaluate(
                    model,
                    test_dataset,
                    vocab,
                    device,
                    max_samples=min(500, len(test_dataset))
                )
                print(f"[TEST EVAL] CRR: {char_acc:.1f}% | Exact Match: {exact_acc:.1f}%")
            else:
                # Fallback to train evaluation if no test directory is available
                char_acc, exact_acc = evaluate(
                    model,
                    dataset,
                    vocab,
                    device,
                    max_samples=min(500, len(dataset))
                )
                print(f"[TRAIN EVAL FALLBACK] CRR: {char_acc:.1f}% | Exact Match: {exact_acc:.1f}%")

            model.eval()
            print("\n--- Sample Predictions (from Train Set) ---")
            for i in range(min(5, len(dataset))):
                sample_img, sample_tgt = dataset[i]
                pred_ids = model.generate(
                    sample_img.unsqueeze(0).to(device),
                    vocab
                )[0].tolist()

                print(f"\nSample {i+1}")
                print("GT   :", vocab.decode(sample_tgt.tolist()))
                print("PRED :", vocab.decode(pred_ids))

            # --- SAVE EVERY 5 EPOCHS ---
            torch.save({
                "epoch": epoch + 1,
                "model_state": model.state_dict(),
            }, f"/kaggle/working/epoch_{epoch+1}.pth")
            print(f"  Checkpoint epoch_{epoch+1}.pth saved")

            # --- SAVE BEST MODEL BASED ON GENERALIZATION PERFORMANCE ---
            if char_acc > best_crr:
                best_crr = char_acc
                best_epoch = epoch + 1  # 5. Track Best Epoch index
                torch.save({
                    "epoch": epoch + 1,
                    "crr": char_acc,
                    "model_state": model.state_dict(),
                    "vocab": vocab,
                    "img_width": img_max_width,
                    "config": config,
                }, "/kaggle/working/best_model.pth")
                print(f" Best model saved (CRR={char_acc:.2f}%)")

            model.train()

    model.eval()
    
    # 4. Inference Time tracking around final sample generation
    sample_img, sample_tgt = dataset[0]
    
    start_inf = time.time()
    pred_ids = model.generate(sample_img.unsqueeze(0).to(device), vocab)[0].tolist()
    end_inf = time.time()
    
    print("\n--- Final Sample Check (train set) ---")
    print("Ground truth :", vocab.decode(sample_tgt.tolist()))
    print("Model output :", vocab.decode(pred_ids))
    print(f"Inference Time: {(end_inf - start_inf):.4f} sec")

    # --- FINAL TEST SET RUN OVER FULL SET ---
    if test_dataset is not None:
        char_acc, exact_acc = evaluate(
            model,
            test_dataset,
            vocab,
            device,
            max_samples=len(test_dataset)
        )
        print(f"\n--- Final Test Set Accuracy (n={len(test_dataset)}) ---")
        print(f"Character accuracy (CRR): {char_acc:.1f}%")
        print(f"Exact line match:         {exact_acc:.1f}%")
    
    torch.save({
        "model_state": model.state_dict(),
        "vocab": vocab,
        "img_width": img_max_width,
        "config": config,
    }, "ocr52.pth")

    # --- VERIFY FINAL SAVE ---
    print("\nModel saved as ocr52.pth")
    print("Exists:", os.path.exists("ocr52.pth"))
    if os.path.exists("ocr52.pth"):
        print("Size:", round(os.path.getsize("ocr52.pth")/(1024*1024),2), "MB")
        
    # 1. Track Total Training Time (End & Output)
    training_end = time.time()
    print("\n====================")
    print("TRAINING TIME")
    print("====================")
    print(f"{(training_end - training_start) / 60:.2f} minutes")
    
    # 5. Summary Final Display for Best Epoch Performance
    print("\n====================")
    print("BEST MODEL")
    print("====================")
    print(f"Best Epoch : {best_epoch}")
    print(f"Best CRR   : {best_crr:.2f}%")
    
    return model, vocab


if __name__ == "__main__":
    model, vocab = train_one_run(CONFIG)

Found 8352 image+text pairs across 8 type(s):
  chaos_mixed: 429
  degraded_pure: 414
  faded_pure: 418
  faded_stain: 426
  large_solid: 1263
  med_solid: 2107
  type1_gaps: 1663
  type2_baseline: 1632
Auto-detected width: your widest line becomes 3920px after resizing height to 48px. Using that.
Vocab size: 74

MODEL INFO
Total Params     : 3,589,898
Trainable Params : 3,589,898
Epoch 1/60  loss=2.6037
Epoch Time: 212.06 sec
Epoch 2/60  loss=2.1889
Epoch Time: 216.12 sec
Epoch 3/60  loss=1.9777
Epoch Time: 215.07 sec
Epoch 4/60  loss=1.6502
Epoch Time: 215.34 sec
Epoch 5/60  loss=1.1912
Epoch Time: 217.19 sec

--- Running Evaluation ---
[TEST EVAL] CRR: 16.8% | Exact Match: 0.0%

--- Sample Predictions (from Train Set) ---

Sample 1
GT   : కాపీలను సభ్యులు సమీక్షించారు తదనంతర పర్వాల ప్రకటన ఇతోధికమగు వేగం అందుకోవటానికి వీలుగా ఈ సమావేశంలో కవిత్రయ భారతానికి సంబంధించిన సంపాదక మండలి జథ
PRED : కాపిలను సహృద్యులు సమీక్షించారు తరువర్తాల పర్వాక ప్రకటి భక్తిక అంతఃపారం వేదం అందువలె ఉండికి వీరికి 

In [1]:
import os

SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

In [2]:
"""
Telugu Manuscript Line Reconstruction - CNN Encoder + Transformer Decoder
===========================================================================
Reads a (possibly damaged) Telugu manuscript line image and outputs the
reconstructed Telugu text. Works across all 8 of your damage types (clean,
gaps, stains, fade, degraded, mixed) since the SAME model is trained on
all of them pooled together - the encoder learns to read through visual
noise, the decoder learns to guess fully-missing spans from context.

HOW TO USE IN KAGGLE:
1. Edit the CONFIG section below to point at your CSV + image folder.
2. Your CSV must have two columns: one with the image filename/path,
   one with the ground-truth Telugu text for that line.
3. Run this script top to bottom. With smoke_test=True it trains on a
   small slice of your data for a few epochs, just to prove the whole
   pipeline runs without crashing and the loss goes down.
4. Once that passes, set smoke_test=False and raise epochs for a real run.
"""

import os
import re
import math
import random
import time
import shutil
import traceback
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T

# --- 1 RANDOM SEED ---
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Define Save Directory Path
SAVE_DIR = "/kaggle/working/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ----------------------------- CONFIG --------------------------------------
CONFIG = {
    # Folder containing your images AND their matching .txt ground-truth
    # files (e.g. chaos_mixed_00003.png + chaos_mixed_00003.txt side by
    # side). If train/test are separate folders, point this at TRAIN.
    "image_dir": "/kaggle/input/datasets/yvenkatasriramreddy/final-telugu-manuscript-data/10k_Data/train",   
    "test_image_dir": "/kaggle/input/datasets/yvenkatasriramreddy/final-telugu-manuscript-data/10k_Data/validation",  
    "image_extensions": (".png", ".jpg", ".jpeg"),
    "text_extension": ".txt",
    "img_height": 48,          # every line image is resized to this height
    "img_max_width": "auto",   # "auto" = measured from YOUR real images, see below
                                # (set a number only if you want to override it)
    "batch_size": 8,           # your lines are very wide (~5300px) -> keep this modest
    "epochs": 60,              # bump this freely while iterating on the smoke subset
    "smoke_test": False,        # True = quick run on a small subset to verify it runs
    "smoke_test_samples": 300,
    "lr": 3e-4,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


# ----------------------------- DATA SCAN ------------------------------------
def scan_dataset(image_dir, image_extensions, text_extension):
    """Walks image_dir (including subfolders, in case you split by type into
    separate folders) and pairs every image with a same-named .txt file."""
    samples = []
    for root, _, files in os.walk(image_dir):
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            if ext not in image_extensions:
                continue
            stem = os.path.splitext(fname)[0]
            txt_path = os.path.join(root, stem + text_extension)
            if not os.path.exists(txt_path):
                continue  # no matching ground-truth file, skip
            with open(txt_path, "r", encoding="utf-8") as f:
                text = f.read().strip()
            if not text:
                continue
            type_tag = re.sub(r"_\d+$", "", stem)  # strip trailing _00003
            samples.append({
                "image_path": os.path.join(root, fname),
                "text": text,
                "type": type_tag,
            })
    return samples


def compute_max_width(samples, img_h):
    """Scans your ACTUAL images to find how wide the widest line becomes once
    every image is resized to img_h."""
    max_w = 0
    for s in samples:
        with Image.open(s["image_path"]) as img:
            w, h = img.size
        new_w = max(1, int(w * (img_h / h)))
        max_w = max(max_w, new_w)
    max_w = ((max_w + 7) // 8) * 8
    return max_w


# ----------------------------- VOCAB ----------------------------------------
class Vocab:
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
        self.itos = specials + chars
        self.stoi = {ch: i for i, ch in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.sos_id = self.stoi["<sos>"]
        self.eos_id = self.stoi["<eos>"]
        self.unk_id = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def encode(self, text):
        ids = [self.sos_id] + [self.stoi.get(ch, self.unk_id) for ch in text] + [self.eos_id]
        return torch.tensor(ids, dtype=torch.long)

    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos_id:
                break
            if i in (self.pad_id, self.sos_id):
                continue
            out.append(self.itos[i])
        return "".join(out)


# ----------------------------- DATASET ---------------------------------------
class TeluguLineDataset(Dataset):
    def __init__(self, samples, vocab, img_h, img_w):
        self.samples = samples
        self.vocab = vocab
        self.img_h = img_h
        self.img_w = img_w
        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.samples)

    def _load_image(self, path):
        img = Image.open(path).convert("L")          # grayscale
        w, h = img.size
        new_w = max(1, int(w * (self.img_h / h)))
        img = img.resize((new_w, self.img_h))
        img = self.to_tensor(img)                      # [1, H, W'] in [0,1]
        c, h_, w_ = img.shape
        if w_ < self.img_w:                             # pad to fixed width
            pad = torch.ones(c, h_, self.img_w - w_)
            img = torch.cat([img, pad], dim=2)
        else:                                            # crop if too wide
            img = img[:, :, : self.img_w]
        return img

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image = self._load_image(sample["image_path"])
        target = self.vocab.encode(sample["text"])
        return image, target


def collate_fn(batch, pad_id):
    images, targets = zip(*batch)
    images = torch.stack(images, dim=0)
    max_len = max(t.size(0) for t in targets)
    padded = torch.full((len(targets), max_len), pad_id, dtype=torch.long)
    for i, t in enumerate(targets):
        padded[i, : t.size(0)] = t
    return images, padded


# ----------------------------- MODEL ------------------------------------------
class CNNEncoder(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, d_model, 3, padding=1), nn.BatchNorm2d(d_model), nn.ReLU(),
        )
        self.height_pool = nn.AdaptiveAvgPool2d((1, None))

    def forward(self, x):
        feat = self.conv(x)            # [B, d_model, H', W']
        feat = self.height_pool(feat)  # [B, d_model, 1, W']
        feat = feat.squeeze(2)         # [B, d_model, W']
        feat = feat.permute(0, 2, 1)   # [B, W', d_model]
        return feat


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TeluguOCRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_decoder_layers=4,
                 dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.encoder = CNNEncoder(d_model)
        self.pos_enc_src = PositionalEncoding(d_model)
        self.tgt_embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc_tgt = PositionalEncoding(d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_decoder_layers)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, images, tgt_in):
        memory = self.pos_enc_src(self.encoder(images))
        tgt_emb = self.pos_enc_tgt(self.tgt_embed(tgt_in))
        T_len = tgt_in.size(1)
        causal_mask = torch.triu(
            torch.ones(T_len, T_len, device=tgt_in.device), diagonal=1
        ).bool()
        out = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)
        return self.fc_out(out)

    @torch.no_grad()
    def generate(self, images, vocab, max_len=512):
        self.eval()
        memory = self.pos_enc_src(self.encoder(images))
        B = images.size(0)
        ys = torch.full((B, 1), vocab.sos_id, dtype=torch.long, device=images.device)
        for _ in range(max_len):
            tgt_emb = self.pos_enc_tgt(self.tgt_embed(ys))
            T_len = ys.size(1)
            causal_mask = torch.triu(
                torch.ones(T_len, T_len, device=images.device), diagonal=1
            ).bool()
            out = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)
            next_id = self.fc_out(out[:, -1, :]).argmax(-1, keepdim=True)
            ys = torch.cat([ys, next_id], dim=1)
            if (next_id == vocab.eos_id).all():
                break
        return ys


def levenshtein(a, b):
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = prev if a[i - 1] == b[j - 1] else 1 + min(prev, dp[j - 1], dp[j])
            prev = cur
    return dp[m]


@torch.no_grad()
def evaluate(model, dataset, vocab, device, max_samples=200):
    model.eval()
    n = min(max_samples, len(dataset))
    indices = random.sample(range(len(dataset)), n)
    total_chars, total_edits, exact = 0, 0, 0
    for idx in indices:
        img, tgt = dataset[idx]
        gt_text = vocab.decode(tgt.tolist())
        pred_ids = model.generate(img.unsqueeze(0).to(device), vocab)[0].tolist()
        pred_text = vocab.decode(pred_ids)
        total_edits += levenshtein(pred_text, gt_text)
        total_chars += max(len(gt_text), 1)
        exact += int(pred_text == gt_text)
    char_acc = 100 * (1 - total_edits / total_chars)
    exact_acc = 100 * exact / n
    return char_acc, exact_acc


# ----------------------------- TRAIN -------------------------------------------
def train_one_run(config):
    samples = scan_dataset(config["image_dir"], config["image_extensions"], config["text_extension"])
    if not samples:
        raise RuntimeError(f"No image+.txt pairs found under {config['image_dir']}.")

    type_counts = {}
    for s in samples:
        type_counts[s["type"]] = type_counts.get(s["type"], 0) + 1
    print(f"Found {len(samples)} image+text pairs across {len(type_counts)} type(s):")
    for t, c in sorted(type_counts.items()):
        print(f"  {t}: {c}")

    if config["smoke_test"]:
        n = min(config["smoke_test_samples"], len(samples))
        samples = random.sample(samples, n)
        print(f"\n[SMOKE TEST] Using {len(samples)} samples.")

    if config["img_max_width"] in (None, "auto"):
        img_max_width = compute_max_width(samples, config["img_height"])
        print(f"Auto-detected width: {img_max_width}px. Using that.")
    else:
        img_max_width = config["img_max_width"]

    vocab = Vocab([s["text"] for s in samples])
    print(f"Vocab size: {len(vocab)}")

    dataset = TeluguLineDataset(samples, vocab, config["img_height"], img_max_width)
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=True,
        collate_fn=lambda b: collate_fn(b, vocab.pad_id),
    )

    test_dataset = None
    if config.get("test_image_dir"):
        test_samples = scan_dataset(config["test_image_dir"], config["image_extensions"], config["text_extension"])
        if test_samples:
            if config["smoke_test"]:
                test_samples = test_samples[: min(100, len(test_samples))]
            test_dataset = TeluguLineDataset(test_samples, vocab, config["img_height"], img_max_width)

    device = config["device"]
    model = TeluguOCRModel(vocab_size=len(vocab)).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print("\n====================")
    print("MODEL INFO")
    print("====================")
    print(f"Total Params     : {total_params:,}")
    print(f"Trainable Params : {trainable_params:,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    criterion = nn.CrossEntropyLoss(ignore_index=vocab.pad_id)

    best_crr = 0
    best_epoch = 0

    training_start = time.time()

    model.train()
    for epoch in range(config["epochs"]):
        epoch_start = time.time()
        total_loss = 0.0
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            tgt_in = targets[:, :-1]
            tgt_out = targets[:, 1:]

            logits = model(images, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{config['epochs']}  loss={total_loss / len(loader):.4f}")
        epoch_end = time.time()
        print(f"Epoch Time: {(epoch_end - epoch_start):.2f} sec")

        if (epoch + 1) % 5 == 0:
            print("\n--- Running Evaluation ---")
            
            # 3. Wrap evaluation loop with robust crash-containment fallbacks
            try:
                if test_dataset is not None:
                    char_acc, exact_acc = evaluate(
                        model, test_dataset, vocab, device, max_samples=min(500, len(test_dataset))
                    )
                    print(f"[TEST EVAL] CRR: {char_acc:.1f}% | Exact Match: {exact_acc:.1f}%")
                else:
                    char_acc, exact_acc = evaluate(
                        model, dataset, vocab, device, max_samples=min(500, len(dataset))
                    )
                    print(f"[TRAIN EVAL FALLBACK] CRR: {char_acc:.1f}% | Exact Match: {exact_acc:.1f}%")
            except Exception:
                print("\n⚠️ EVALUATION CRASHED! Catching and writing panic snapshot to storage...")
                traceback.print_exc()
                checkpoint_path = os.path.join(SAVE_DIR, f"crash_epoch_{epoch+1}.pth")
                torch.save({"epoch": epoch + 1, "model_state": model.state_dict()}, checkpoint_path)
                os.sync()
                raise

            # 2. SAVE CHECKPOINTS BEFORE PRINT SAMPLES PREDICTIONS LOOP
            checkpoint_path = os.path.join(SAVE_DIR, f"epoch_{epoch+1}.pth")
            torch.save({"epoch": epoch + 1, "model_state": model.state_dict()}, checkpoint_path)
            os.sync()  # Force allocation writes
            print(f"✅ Saved: {checkpoint_path}")
            print("Exists:", os.path.exists(checkpoint_path))

            if char_acc > best_crr:
                best_crr = char_acc
                best_epoch = epoch + 1
                
                best_path = os.path.join(SAVE_DIR, "best_model.pth")
                torch.save({
                    "epoch": epoch + 1, "crr": char_acc, "model_state": model.state_dict(),
                    "vocab": vocab, "img_width": img_max_width, "config": config,
                }, best_path)
                os.sync()
                print(f"🔥 BEST MODEL SAVED (Epoch {epoch+1})")
                print("Exists:", os.path.exists(best_path))

            # Sample printing loop executed safely after checkpoint disk storage
            model.eval()
            print("\n--- Sample Predictions (from Train Set) ---")
            for i in range(min(5, len(dataset))):
                sample_img, sample_tgt = dataset[i]
                pred_ids = model.generate(sample_img.unsqueeze(0).to(device), vocab)[0].tolist()
                print(f"\nSample {i+1}")
                print("GT   :", vocab.decode(sample_tgt.tolist()))
                print("PRED :", vocab.decode(pred_ids))

            model.train()

    model.eval()
    sample_img, sample_tgt = dataset[0]
    
    start_inf = time.time()
    pred_ids = model.generate(sample_img.unsqueeze(0).to(device), vocab)[0].tolist()
    end_inf = time.time()
    
    print("\n--- Final Sample Check (train set) ---")
    print("Ground truth :", vocab.decode(sample_tgt.tolist()))
    print("Model output :", vocab.decode(pred_ids))
    print(f"Inference Time: {(end_inf - start_inf):.4f} sec")

    if test_dataset is not None:
        char_acc, exact_acc = evaluate(model, test_dataset, vocab, device, max_samples=len(test_dataset))
        print(f"\n--- Final Test Set Accuracy (n={len(test_dataset)}) ---")
        print(f"Character accuracy (CRR): {char_acc:.1f}%")
        print(f"Exact line match:         {exact_acc:.1f}%")
    
    # Final path with accuracy context appended
    final_path = os.path.join(SAVE_DIR, f"ocr_epoch{config['epochs']}_{best_crr:.2f}.pth")
    torch.save({
        "epoch": config["epochs"], "crr": best_crr, "model_state": model.state_dict(),
        "vocab": vocab, "img_width": img_max_width, "config": config,
    }, final_path)
    os.sync()

    print("\n====================")
    print("FINAL MODEL SAVED")
    print("====================")
    print(final_path)
    print("Exists:", os.path.exists(final_path))
    print("Size:", round(os.path.getsize(final_path)/(1024*1024),2), "MB")
        
    training_end = time.time()
    print("\n====================")
    print("TRAINING TIME")
    print("====================")
    print(f"{(training_end - training_start) / 60:.2f} minutes")
    
    print("\n====================")
    print("BEST MODEL")
    print("====================")
    print(f"Best Epoch : {best_epoch}")
    print(f"Best CRR   : {best_crr:.2f}%")
    
    print("\n====================")
    print("ALL SAVED FILES")
    print("====================")
    for f in sorted(os.listdir(SAVE_DIR)):
        if f.endswith(".pth"):
            size = os.path.getsize(os.path.join(SAVE_DIR, f)) / (1024 * 1024)
            print(f"{f:20s} {size:.2f} MB")
    
    return model, vocab


if __name__ == "__main__":
    try:
        model, vocab = train_one_run(CONFIG)
    except Exception:
        print("\n" + "="*80)
        print("TRAINING CRASHED")
        print("="*80)
        traceback.print_exc()
        
        print("\nFiles currently saved:")
        for f in sorted(os.listdir(SAVE_DIR)):
            if f.endswith(".pth"):
                print(f)
    finally:
        # 5. Compress full run output workspace context cleanly into single ZIP target
        print("\nPacking output directory context to a single archive...")
        try:
            shutil.make_archive("/kaggle/working/OCR_Backup", "zip", SAVE_DIR)
            print("✅ Backup ZIP created successfully: OCR_Backup.zip")
        except Exception as zip_err:
            print(f"⚠️ Failed to pack backup ZIP archive: {zip_err}")

Found 8352 image+text pairs across 8 type(s):
  chaos_mixed: 429
  degraded_pure: 414
  faded_pure: 418
  faded_stain: 426
  large_solid: 1263
  med_solid: 2107
  type1_gaps: 1663
  type2_baseline: 1632
Auto-detected width: 3920px. Using that.
Vocab size: 74

MODEL INFO
Total Params     : 3,589,898
Trainable Params : 3,589,898
Epoch 1/60  loss=2.6037
Epoch Time: 224.13 sec
Epoch 2/60  loss=2.1880
Epoch Time: 225.73 sec
Epoch 3/60  loss=1.9764
Epoch Time: 225.76 sec
Epoch 4/60  loss=1.6529
Epoch Time: 227.45 sec
Epoch 5/60  loss=1.2192
Epoch Time: 227.19 sec

--- Running Evaluation ---
[TEST EVAL] CRR: 24.2% | Exact Match: 0.0%
✅ Saved: /kaggle/working/epoch_5.pth
Exists: True
🔥 BEST MODEL SAVED (Epoch 5)
Exists: True

--- Sample Predictions (from Train Set) ---

Sample 1
GT   : కాపీలను సభ్యులు సమీక్షించారు తదనంతర పర్వాల ప్రకటన ఇతోధికమగు వేగం అందుకోవటానికి వీలుగా ఈ సమావేశంలో కవిత్రయ భారతానికి సంబంధించిన సంపాదక మండలి జథ
PRED : కానిపోలను సహృతులు సమీచారు తరువంశం తరువాత ప్రకటి అంక దంకోరుకొన

In [3]:
import os

# 1. Force list everything inside /kaggle/working
files = os.listdir('/kaggle/working')
print("--- Files found in /kaggle/working ---")
for file in sorted(files):
    full_path = os.path.join('/kaggle/working', file)
    size_mb = os.path.getsize(full_path) / (1024 * 1024)
    print(f"📄 {file} ({size_mb:.2f} MB)")

# 2. Check if the final model file exists explicitly
print("\nExplicit file verification:")
print("best_model.pth exists:", os.path.exists("/kaggle/working/best_model.pth"))

--- Files found in /kaggle/working ---
📄 .virtual_documents (0.00 MB)

Explicit file verification:
best_model.pth exists: False


In [1]:
import os

print("Contents of /kaggle/working:\n")

files = os.listdir("/kaggle/working")

print(files)
print("\nTotal files:", len(files))

Contents of /kaggle/working:

['.virtual_documents']

Total files: 1


In [4]:
import os
import torch

# 1. Figure out EXACTLY where Python thinks it is right now
print("Current Working Directory:", os.getcwd())

# 2. Re-verify root path accessibility
TARGET_DIR = "/kaggle/working"
print(f"Is {TARGET_DIR} writable?:", os.access(TARGET_DIR, os.W_OK))

# 3. Create a test dummy file explicitly to see if it shows up
test_path = os.path.join(TARGET_DIR, "force_test.txt")
with open(test_path, "w") as f:
    f.write("System test")

print("Did dummy save exist?:", os.path.exists(test_path))
print("Updated directory list:", os.listdir(TARGET_DIR))

Current Working Directory: /kaggle/working
Is /kaggle/working writable?: True
Did dummy save exist?: True
Updated directory list: ['force_test.txt', '.virtual_documents']


In [8]:
import torch

ckpt = torch.load(
    "/kaggle/working/best_model.pth",
    map_location="cpu",
    weights_only=False
)

print(type(ckpt))
print(ckpt.keys())

<class 'dict'>
dict_keys(['epoch', 'crr', 'model_state', 'vocab', 'img_width', 'config'])


In [36]:
import time

img, tgt = val_dataset[0]

start = time.time()

pred_ids = model.generate(
    img.unsqueeze(0).to(DEVICE),
    vocab,
    max_len=256
)

end = time.time()

print("Time:", end - start)
print("Generated tokens:", pred_ids.shape[1])

print("\nPrediction:")
print(vocab.decode(pred_ids[0].tolist()))

print("\nGround Truth:")
print(vocab.decode(tgt.tolist()))

/tmp/ipykernel_58/1364502190.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt = torch.tensor(


Time: 0.8058841228485107
Generated tokens: 189

Prediction:
తల్లిదండ్రులందు మిక్కిలి వినయ విధేయతలు ప్రతర్శించెను శ్రీరాముడు భరద్వాజుని ఆశ్రవుమునకు వెళ్లినపుడు అత్రి మహర్షి ఆశ్రమమునకు శరభంగుని ఆశ్రమమునకు సుతీష్టుని ఆశ్రమమునకు వెళ్లినపుడు అగస్త్యుని

Ground Truth:
తల్లిదండ్రులందు మిక్కిలి వినయ విధేయతలు ప్రదర్శించెను శ్రీరాముడు భరద్వాజుని ఆశ్రవుమునకు వెళ్లినపుడు అత్రి మహర్షి ఆశ్రమమునకు శరభంగుని ఆశ్రమమునకు సుతీష్టుని ఆశ్రమమునకు వెళ్లినపుడు అగస్త్యుని


In [41]:
import random
import time
import torch

@torch.no_grad()
def evaluate(model, dataset, vocab, device, max_samples=None, progress_every=50):

    model.eval()

    if max_samples is None:
        n = len(dataset)
    else:
        n = min(max_samples, len(dataset))

    indices = random.sample(range(len(dataset)), n)

    total_chars = 0
    total_edits = 0
    exact = 0

    start_time = time.time()

    print("=" * 70)
    print("STARTING EVALUATION")
    print("=" * 70)
    print(f"Total Images : {n}\n")

    for i, idx in enumerate(indices, start=1):

        img, tgt = dataset[idx]

        gt_text = vocab.decode(tgt.tolist())

        pred_ids = model.generate(
            img.unsqueeze(0).to(device),
            vocab,
            max_len=256
        )[0].tolist()

        pred_text = vocab.decode(pred_ids)

        total_edits += levenshtein(pred_text, gt_text)
        total_chars += max(len(gt_text), 1)
        exact += int(pred_text == gt_text)

        if i % progress_every == 0 or i == n:

            elapsed = time.time() - start_time
            avg = elapsed / i
            remaining = avg * (n - i)

            print(
                f"[{i:4d}/{n}]  "
                f"Elapsed: {elapsed:.1f}s  "
                f"ETA: {remaining:.1f}s"
            )

    total_time = time.time() - start_time

    char_acc = 100 * (1 - total_edits / total_chars)
    exact_acc = 100 * exact / n

    print("\n" + "=" * 70)
    print("EVALUATION COMPLETED")
    print("=" * 70)

    print(f"Images            : {n}")
    print(f"CRR               : {char_acc:.2f}%")
    print(f"Exact Match       : {exact_acc:.2f}%")
    print(f"Total Time        : {total_time:.2f} sec")
    print(f"Per Image         : {total_time/n:.4f} sec")

    return char_acc, exact_acc

In [42]:
crr, exact = evaluate(
    model=model,
    dataset=val_dataset,
    vocab=vocab,
    device=DEVICE,
    max_samples=None,
    progress_every=50
)

STARTING EVALUATION
Total Images : 987



/tmp/ipykernel_58/1364502190.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt = torch.tensor(


[  50/987]  Elapsed: 37.1s  ETA: 695.8s
[ 100/987]  Elapsed: 73.8s  ETA: 654.2s
[ 150/987]  Elapsed: 110.9s  ETA: 619.1s
[ 200/987]  Elapsed: 147.0s  ETA: 578.6s
[ 250/987]  Elapsed: 183.3s  ETA: 540.5s
[ 300/987]  Elapsed: 220.1s  ETA: 504.0s
[ 350/987]  Elapsed: 256.1s  ETA: 466.1s
[ 400/987]  Elapsed: 292.9s  ETA: 429.9s
[ 450/987]  Elapsed: 329.3s  ETA: 393.0s
[ 500/987]  Elapsed: 365.7s  ETA: 356.2s
[ 550/987]  Elapsed: 402.3s  ETA: 319.6s
[ 600/987]  Elapsed: 438.7s  ETA: 282.9s
[ 650/987]  Elapsed: 474.9s  ETA: 246.2s
[ 700/987]  Elapsed: 511.6s  ETA: 209.8s
[ 750/987]  Elapsed: 547.9s  ETA: 173.2s
[ 800/987]  Elapsed: 584.4s  ETA: 136.6s
[ 850/987]  Elapsed: 621.6s  ETA: 100.2s
[ 900/987]  Elapsed: 658.0s  ETA: 63.6s
[ 950/987]  Elapsed: 694.8s  ETA: 27.1s
[ 987/987]  Elapsed: 722.4s  ETA: 0.0s

EVALUATION COMPLETED
Images            : 987
CRR               : 96.18%
Exact Match       : 37.99%
Total Time        : 722.41 sec
Per Image         : 0.7319 sec


In [44]:
test_samples = scan_dataset(
    "/kaggle/input/datasets/yvenkatasriramreddy/final-telugu-manuscript-data/10k_Data/test",
    CONFIG["image_extensions"],
    CONFIG["text_extension"]
)

print("Test Images:", len(test_samples))

Test Images: 980


In [45]:
test_dataset = OCRDataset(
    samples=test_samples,
    vocab=vocab,
    img_h=CONFIG["img_height"],
    img_w=ckpt["img_width"]
)

In [46]:
crr, exact = evaluate(
    model=model,
    dataset=test_dataset,
    vocab=vocab,
    device=DEVICE,
    max_samples=None,
    progress_every=50
)

STARTING EVALUATION
Total Images : 980



/tmp/ipykernel_58/1364502190.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tgt = torch.tensor(


[  50/980]  Elapsed: 37.4s  ETA: 696.0s
[ 100/980]  Elapsed: 75.4s  ETA: 663.9s
[ 150/980]  Elapsed: 112.9s  ETA: 624.9s
[ 200/980]  Elapsed: 150.6s  ETA: 587.5s
[ 250/980]  Elapsed: 188.7s  ETA: 551.1s
[ 300/980]  Elapsed: 225.4s  ETA: 511.0s
[ 350/980]  Elapsed: 262.9s  ETA: 473.2s
[ 400/980]  Elapsed: 299.7s  ETA: 434.5s
[ 450/980]  Elapsed: 336.3s  ETA: 396.1s
[ 500/980]  Elapsed: 374.4s  ETA: 359.4s
[ 550/980]  Elapsed: 411.9s  ETA: 322.0s
[ 600/980]  Elapsed: 449.5s  ETA: 284.7s
[ 650/980]  Elapsed: 487.1s  ETA: 247.3s
[ 700/980]  Elapsed: 523.8s  ETA: 209.5s
[ 750/980]  Elapsed: 560.3s  ETA: 171.8s
[ 800/980]  Elapsed: 597.1s  ETA: 134.3s
[ 850/980]  Elapsed: 634.6s  ETA: 97.1s
[ 900/980]  Elapsed: 671.8s  ETA: 59.7s
[ 950/980]  Elapsed: 708.7s  ETA: 22.4s
[ 980/980]  Elapsed: 730.6s  ETA: 0.0s

EVALUATION COMPLETED
Images            : 980
CRR               : 96.33%
Exact Match       : 41.53%
Total Time        : 730.56 sec
Per Image         : 0.7455 sec
